# Colab single-model runner (Exp1-Exp5) — BioMistral

This notebook runs one judge model end-to-end with repository adapters and a 3-stage fallback pipeline (batch, per-item retry, hard-split) to reduce NA scores and preserve rationales.

Outputs:
- `exp1_dataset_analysis.json`
- `exp2_single_model_detailed.json`
- `exp3_single_model_sensitivity.json`
- `exp4_single_model_plots/*.png`
- `exp5_quality_report.json`


In [1]:
import os
from pathlib import Path

# Configuration
JUDGE_NAME = 'biomistral'
MODEL_ID = 'BioMistral/BioMistral-7B'
MAX_ROWS = 100  # set to None for full dataset
CHECKPOINT_EVERY = 5
MAX_INPUT_TOKENS = 3072
USE_4BIT = True
SEED = 42

REPO_URL = 'https://github.com/mmm-byte/Multi_LLM_Medical_AI_Judge_copy.git'
REPO_NAME = 'Multi_LLM_Medical_AI_Judge_copy'
BASE_DIR = '/content' if Path('/content').exists() and os.access('/content', os.W_OK) else '/tmp/copilot_colab'
REPO_DIR = f'{BASE_DIR}/{REPO_NAME}'
DATASET_CANDIDATES = [
    f'{REPO_DIR}/benchmark_dataset/1000_questions_dataset.csv',
    f'{REPO_DIR}/benchmark_dataset/agreement_benchmark.csv',
    f'{REPO_DIR}/benchmark_dataset/source_datasets/benchmark_dataset_500.csv',
]
OUTPUT_ROOT = f'{BASE_DIR}/judge_outputs'
RUN_DIR = f'{OUTPUT_ROOT}/{JUDGE_NAME}'
OUTPUT_CSV = f'{RUN_DIR}/{JUDGE_NAME}_item_scores.csv'

# Optional runtime overrides for testing/CI
JUDGE_NAME = os.environ.get('JUDGE_NAME_OVERRIDE', JUDGE_NAME)
MODEL_ID = os.environ.get('MODEL_ID_OVERRIDE', MODEL_ID)
max_rows_env = os.environ.get('MAX_ROWS_OVERRIDE')
if max_rows_env:
    MAX_ROWS = None if max_rows_env.lower() == 'none' else int(max_rows_env)
RUN_DIR = f'{OUTPUT_ROOT}/{JUDGE_NAME}'
OUTPUT_CSV = f'{RUN_DIR}/{JUDGE_NAME}_item_scores.csv'
print(JUDGE_NAME, '->', MODEL_ID, '| BASE_DIR:', BASE_DIR)


biomistral -> BioMistral/BioMistral-7B | BASE_DIR: /tmp/copilot_colab


In [2]:
# Setup
%pip -q install -U transformers accelerate bitsandbytes pandas matplotlib seaborn
!mkdir -p "{BASE_DIR}"
!test -d "{REPO_DIR}" || git clone "{REPO_URL}" "{REPO_DIR}"
%cd "{REPO_DIR}"

import os
import re
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import sys
sys.path.insert(0, REPO_DIR)
from core.model_adapters import get_adapter, ENDPOINT_CHAT
from core.consensus_core.models import RubricItem
from core.rubric_engine import DynamicRubricParser

random.seed(SEED)
os.makedirs(RUN_DIR, exist_ok=True)
Path(RUN_DIR, 'exp4_single_model_plots').mkdir(parents=True, exist_ok=True)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
/Users/mahindragupthakotha/Git Repo/Multi_LLM_Medical_AI_Judge_copy


/Users/mahindragupthakotha/.pyenv/versions/3.12.3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Dataset + rubrics
dataset_path = None
for p in DATASET_CANDIDATES:
    if Path(p).exists():
        dataset_path = p
        break
if dataset_path is None:
    raise FileNotFoundError(f'No dataset found in candidates: {DATASET_CANDIDATES}')

df = pd.read_csv(dataset_path, dtype=str).fillna('')
col_map = {}
for c in df.columns:
    lc = c.lower()
    if lc in ('question', 'text', 'prompt', 'input'):
        col_map[c] = 'question'
    elif lc in ('answer', 'reference_answer', 'output', 'response'):
        col_map[c] = 'answer'
    elif lc in ('question_id',):
        col_map[c] = 'id'
    elif lc in ('source_dataset', 'source'):
        col_map[c] = 'source_dataset'
df = df.rename(columns=col_map)
if 'id' not in df.columns:
    df['id'] = [f'q_{i:05d}' for i in range(len(df))]
if 'domain' not in df.columns:
    df['domain'] = 'unknown'
if 'source_dataset' not in df.columns:
    df['source_dataset'] = Path(dataset_path).stem
required = {'id', 'question', 'answer', 'domain', 'source_dataset'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Dataset missing required columns: {sorted(missing)}')
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

rubric_paths = sorted(Path(REPO_DIR, 'rubrics').glob('rubric*.json'))
rubrics = [json.loads(p.read_text()) for p in rubric_paths]
if len(rubrics) < 1:
    raise ValueError('No rubric JSON files found')

print('Dataset:', dataset_path)
print('Rows:', len(df), '| Rubrics:', len(rubrics), '| Evaluations:', len(df) * len(rubrics))

Dataset: /tmp/copilot_colab/Multi_LLM_Medical_AI_Judge_copy/benchmark_dataset/1000_questions_dataset.csv
Rows: 100 | Rubrics: 5 | Evaluations: 500


In [8]:
# Load model
token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACEHUB_API_TOKEN')
if 'medgemma' in MODEL_ID and not token:
    print('NOTE: MedGemma is gated; set HF_TOKEN in Colab secrets/environment if loading fails.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=token)
load_kwargs = {'trust_remote_code': True, 'device_map': 'auto', 'token': token}
if USE_4BIT and torch.cuda.is_available():
    load_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
else:
    load_kwargs['torch_dtype'] = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
adapter = get_adapter(JUDGE_NAME)
print('Loaded:', MODEL_ID)
print('Adapter:', adapter.__class__.__name__, '| SKIP_BATCH:', getattr(adapter, 'SKIP_BATCH', False))

KeyboardInterrupt: 

In [ ]:
# Inference + parsing helpers with 3-stage fallback
NA_TOKENS = {'', 'NA', 'N/A', 'NONE', 'NULL'}

def is_na(v):
    return str(v).strip().upper() in NA_TOKENS

def to_items(rubric):
    return [
        RubricItem(
            id=it['id'],
            name=it['name'],
            description=it['description'],
            scale=it['scale'],
            weight=float(it.get('weight', 1.0)),
            source_paper='',
        )
        for it in rubric['items']
    ]

def messages_to_prompt(messages, endpoint):
    if endpoint == ENDPOINT_CHAT:
        if hasattr(tokenizer, 'apply_chat_template') and getattr(tokenizer, 'chat_template', None):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return '\n'.join([f"{m.get('role', 'user').upper()}: {m.get('content', '')}" for m in messages]) + '\nASSISTANT:'
    msg0 = messages[0]
    return msg0.get('prompt') or msg0.get('content') or str(msg0)

def generate_text(prompt, max_new_tokens, stop=None):
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    toks = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_TOKENS)
    toks = {k: v.to(next(model.parameters()).device) for k, v in toks.items()}
    with torch.inference_mode():
        out = model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_ids = out[0, toks['input_ids'].shape[1]:]
    txt = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    if stop:
        cut = len(txt)
        for s in stop:
            if s:
                idx = txt.find(s)
                if idx >= 0:
                    cut = min(cut, idx)
        txt = txt[:cut].strip()
    return txt

def evaluate_pair(row, rubric):
    items = to_items(rubric)
    item_by_id = {it.id: it for it in items}
    parser = DynamicRubricParser(type('R', (), {'items': items, 'name': rubric['name']})())
    stage_raw = {'stage1': '', 'stage2': {}, 'stage3': {}}

    item_scores = {it.id: {'score': 'NA', 'rationale': '(unfilled)'} for it in items}

    if not getattr(adapter, 'SKIP_BATCH', False):
        msgs = adapter.build_messages(items, str(row['question']), str(row['answer']))
        prompt = messages_to_prompt(msgs, getattr(adapter, 'ENDPOINT', 'completion'))
        p = adapter.extra_params()
        raw = generate_text(prompt, int(p.get('max_tokens', 256)), p.get('stop'))
        stage_raw['stage1'] = raw
        parsed = adapter.parse(raw, items, JUDGE_NAME)
        for sc in parsed:
            item_scores[sc.rubric_item_id] = {'score': str(sc.score), 'rationale': str(sc.rationale or '')}

    for it in items:
        if is_na(item_scores[it.id]['score']):
            msgs = adapter.build_item_messages(it, str(row['question']), str(row['answer']))
            endpoint = getattr(adapter, 'ITEM_ENDPOINT', getattr(adapter, 'ENDPOINT', 'completion'))
            prompt = messages_to_prompt(msgs, endpoint)
            p = adapter.extra_params_item()
            raw = generate_text(prompt, int(p.get('max_tokens', 64)), p.get('stop'))
            stage_raw['stage2'][it.id] = raw
            sc = adapter.parse_item(raw, it, JUDGE_NAME)
            if not is_na(sc.score):
                item_scores[it.id] = {'score': str(sc.score), 'rationale': str(sc.rationale or '')}

    for it in items:
        if is_na(item_scores[it.id]['score']):
            msgs = adapter.build_split_item_messages(it, str(row['question']), str(row['answer']))
            endpoint = getattr(adapter, 'ITEM_ENDPOINT', getattr(adapter, 'ENDPOINT', 'completion'))
            prompt = messages_to_prompt(msgs, endpoint)
            p = adapter.extra_params_split()
            raw = generate_text(prompt, int(p.get('max_tokens', 16)), p.get('stop'))
            stage_raw['stage3'][it.id] = raw
            sc = adapter.parse_item(raw, it, JUDGE_NAME)
            if not is_na(sc.score):
                item_scores[it.id] = {'score': str(sc.score), 'rationale': str(sc.rationale or '')}

    for it in items:
        if not item_scores[it.id]['rationale']:
            item_scores[it.id]['rationale'] = '(parsed-score-no-rationale)'

    agg_inputs = []
    for it in items:
        x = item_scores[it.id]
        agg_inputs.append(type('S', (), {'rubric_item_id': it.id, 'score': x['score'], 'rationale': x['rationale']})())
    aggregate = parser.aggregate_score(agg_inputs)
    na_count = sum(1 for it in items if is_na(item_scores[it.id]['score']))
    return item_scores, stage_raw, aggregate, na_count


In [ ]:
# Run evaluation with checkpoints
columns = [
    'question_id', 'domain', 'source_dataset', 'question', 'answer',
    'rubric_id', 'rubric_name', 'judge_name', 'model_id', 'item_id', 'score', 'rationale',
    'aggregate_score', 'na_count', 'raw_stage1', 'raw_stage2_json', 'raw_stage3_json'
]
rows = []
done = set()
if Path(OUTPUT_CSV).exists():
    prev = pd.read_csv(OUTPUT_CSV, dtype=str).fillna('')
    rows = prev.to_dict('records')
    done = set((r['question_id'], r['rubric_id']) for r in rows)
    print('Resuming existing rows:', len(rows), '| completed pairs:', len(done))

for i, row in df.iterrows():
    for rubric in rubrics:
        qid = str(row['id'])
        rid = rubric['id']
        if (qid, rid) in done:
            continue
        item_scores, stage_raw, agg, na_count = evaluate_pair(row, rubric)
        for it in rubric['items']:
            rec = {
                'question_id': qid,
                'domain': str(row['domain']),
                'source_dataset': str(row['source_dataset']),
                'question': str(row['question']),
                'answer': str(row['answer']),
                'rubric_id': rid,
                'rubric_name': rubric['name'],
                'judge_name': JUDGE_NAME,
                'model_id': MODEL_ID,
                'item_id': it['id'],
                'score': item_scores[it['id']]['score'],
                'rationale': item_scores[it['id']]['rationale'],
                'aggregate_score': round(float(agg), 4),
                'na_count': int(na_count),
                'raw_stage1': stage_raw['stage1'],
                'raw_stage2_json': json.dumps(stage_raw['stage2']),
                'raw_stage3_json': json.dumps(stage_raw['stage3']),
            }
            rows.append(rec)
        done.add((qid, rid))
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows, columns=columns).to_csv(OUTPUT_CSV, index=False)
        print(f'Checkpoint: {i+1}/{len(df)} questions -> {OUTPUT_CSV}')

result_df = pd.DataFrame(rows, columns=columns)
result_df.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print('Rows:', len(result_df), '| Unique question-rubric pairs:', result_df[['question_id','rubric_id']].drop_duplicates().shape[0])

In [ ]:
# Build Exp1-Exp5 artifacts
exp_dir = Path(RUN_DIR, 'artifacts')
exp_dir.mkdir(parents=True, exist_ok=True)

# Exp1
exp1 = {
    'experiment': 'exp1_dataset_analysis',
    'n_questions': int(len(df)),
    'domain_counts': df['domain'].value_counts().to_dict(),
    'source_counts': df['source_dataset'].value_counts().to_dict(),
    'dataset_path': dataset_path,
}
Path(exp_dir, 'exp1_dataset_analysis.json').write_text(json.dumps(exp1, indent=2))

# Exp2
exp2 = []
for (rid, rname), g in result_df.groupby(['rubric_id', 'rubric_name'], sort=False):
    rows2 = []
    for qid, qg in g.groupby('question_id', sort=False):
        rows2.append({
            'question_id': qid,
            'question_category': qg['domain'].iloc[0],
            'aggregate_score': float(qg['aggregate_score'].iloc[0]),
            'na_count': int(float(qg['na_count'].iloc[0])),
            'item_scores': qg[['item_id','score','rationale']].to_dict('records'),
        })
    exp2.append({'rubric_id': rid, 'rubric_name': rname, 'results': rows2})
Path(exp_dir, 'exp2_single_model_detailed.json').write_text(json.dumps(exp2, indent=2))

# Exp3
def to_num(x):
    try:
        return float(x)
    except Exception:
        return None
exp3 = []
for (rid, rname), g in result_df.groupby(['rubric_id', 'rubric_name'], sort=False):
    vals = [to_num(x) for x in g['score']]
    vals = [x for x in vals if x is not None]
    exp3.append({
        'rubric_id': rid,
        'rubric_name': rname,
        'n_scored_items': len(vals),
        'mean_score': round(sum(vals) / len(vals), 4) if vals else None,
        'score_std': round(float(pd.Series(vals).std()), 4) if len(vals) > 1 else 0.0,
    })
Path(exp_dir, 'exp3_single_model_sensitivity.json').write_text(json.dumps(exp3, indent=2))

# Exp4
plot_df = result_df.copy()
plot_df['score_num'] = pd.to_numeric(plot_df['score'], errors='coerce')
plot_df = plot_df.dropna(subset=['score_num'])
plot_dir = Path(RUN_DIR, 'exp4_single_model_plots')
for rid, g in plot_df.groupby('rubric_id'):
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=g, x='domain', y='score_num')
    plt.title(f'{JUDGE_NAME} | {rid}')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(plot_dir / f'exp4_{rid}.png', dpi=180)
    plt.close()

# Exp5
group_pairs = result_df.groupby(['question_id','rubric_id'], sort=False)
na_item_rate = float((result_df['score'].astype(str).str.upper().isin(['NA','N/A',''])).mean())
zero_rationale_rate = float((result_df['rationale'].astype(str).str.strip() == '').mean())
exp5 = {
    'experiment': 'exp5_quality_report',
    'judge_name': JUDGE_NAME,
    'model_id': MODEL_ID,
    'n_questions': int(result_df['question_id'].nunique()),
    'n_rubrics': int(result_df['rubric_id'].nunique()),
    'n_pairs': int(group_pairs.ngroups),
    'n_item_rows': int(len(result_df)),
    'na_item_rate': round(na_item_rate, 6),
    'empty_rationale_rate': round(zero_rationale_rate, 6),
    'mean_na_items_per_pair': round(float(pd.to_numeric(result_df['na_count'], errors='coerce').fillna(0).mean()), 4),
}
Path(exp_dir, 'exp5_quality_report.json').write_text(json.dumps(exp5, indent=2))

print('Artifacts written to:', exp_dir)
print(json.dumps(exp5, indent=2))

In [ ]:
# Quick validation checks
pair_count = result_df[['question_id','rubric_id']].drop_duplicates().shape[0]
expected_pairs = len(df) * len(rubrics)
print('Pairs:', pair_count, '/', expected_pairs)
if pair_count != expected_pairs:
    raise RuntimeError('Missing evaluated question-rubric pairs; rerun notebook from checkpoint cell.')

na_items = int(result_df['score'].astype(str).str.upper().isin(['NA', 'N/A', '']).sum())
print('NA item rows:', na_items, '/', len(result_df))
display(result_df.head(10))